In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import auc

def analyze_risk_coverage(confidences: np.ndarray, predictions: np.ndarray, true_labels: np.ndarray, target_risk: float = 0.05):
    """
    Sweeps confidence thresholds to plot the risk-coverage curve and calculate AURC[cite: 2].
    Finds the optimal operating threshold that bounds clinical risk.
    """
    thresholds = np.linspace(0.0, 1.0, 100)
    coverages = []
    risks = []
    
    for t in thresholds:
        # Cases the model is confident enough to auto-report
        covered_indices = np.where(confidences >= t)[0]
        coverage = len(covered_indices) / len(confidences)
        coverages.append(coverage)
        
        if coverage > 0:
            # Error rate among the auto-reported cases
            retained_preds = predictions[covered_indices]
            retained_labels = true_labels[covered_indices]
            error_rate = 1.0 - np.mean(retained_preds == retained_labels)
            risks.append(error_rate)
        else:
            risks.append(0.0)
            
    # Calculate Area Under the Risk-Coverage Curve (AURC)[cite: 2]
    aurc = auc(coverages, risks)
    
    # Find the lowest confidence threshold that satisfies the target risk bound
    valid_thresholds = [t for t, r in zip(thresholds, risks) if r <= target_risk]
    optimal_threshold = valid_thresholds[0] if valid_thresholds else 1.0
    
    # Plotting the curve
    plt.figure(figsize=(8, 6))
    plt.plot(coverages, risks, marker='.', label=f'Model Performance (AURC: {aurc:.4f})')
    plt.axhline(y=target_risk, color='r', linestyle='--', label=f'Target Risk Bound ({target_risk*100}%)')
    plt.xlabel('Coverage (Fraction of cases auto-reported)')
    plt.ylabel('Risk (Error rate of reported cases)')
    plt.title('Risk-Coverage Curve for Clinical Deferral')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    print(f"Optimal Confidence Threshold to maintain <= {target_risk*100}% risk: {optimal_threshold:.4f}")
    return optimal_threshold, aurc